# Train on ALS classification and then compute Pr(TIC) of a separate subset (and viceversa)

First of all we need to define two datasets, one for training the ALS model (and testing the TIC one) and one for training TIC and testing ALS model.

We want all classes to have the same #images, but the control untreated images in the two datasets can not be overlapping.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import yaml
import pickle
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.neural_network import MLPClassifier
from utils.mypath import MyPath
from utils.mypalette import day_palette, als_palette, condition_palette
from utils.start_notebook import setup_experiment
from utils.crosstask_utils import train_MLP_classifier, subsample_data
import warnings
import torch

warnings.filterwarnings("ignore")
# Set random seed for reproducibility
np.random.seed(42)

x = '240813_5'

batches = ['E4']

classifiers = ['ALS', 'TIChigh', 'TIClow']

res_dir = '../results'
config_dir = '../configs'
saving_dir = os.path.join(res_dir, 'cross_task_classification')

if not os.path.exists(saving_dir):
    os.makedirs(saving_dir)

if not os.path.exists(os.path.join(saving_dir, 'regular')):
    os.makedirs(os.path.join(saving_dir, 'regular'))

if not os.path.exists(os.path.join(saving_dir, 'cross_test')):
    os.makedirs(os.path.join(saving_dir, 'cross_test'))


###
# LOAD EMBEDDINGS AND ANNOTATIONS, BALANCE DATASET MAKING SURE THAT CTRL UNTR SUBSETS ARE NOT OVERLAPPING
###
annotations, raw_emb = setup_experiment(x, batches)
annotations['simclr_embedding'] = list(raw_emb)

In [ ]:
for b in batches:

    batch_ann = annotations[annotations['experiment'] == b]
    validation_data_dict = subsample_data(batch_ann)

    for c in classifiers:

        data = validation_data_dict[c]

        if c in ['TIChigh', 'TIClow']:
            hue = 'condition'
        if c == 'ALS':
            data['als_label'] = data['als_label'].replace({'als': 'ALS', 'control': 'CTRL'})
            hue='als_label'

        saving_path = os.path.join(saving_dir, 'regular', b)
        os.makedirs(saving_path, exist_ok=True)

        best_model, best_fold, best_auc, scores, folds_annotations = train_MLP_classifier(c, data, "simclr_embedding", saving_path)

        print(f'The best AUC for the {c.replace("C", "C ")} classifier is {best_auc}, achieved with fold {best_fold}')
        print(f'The AUC scores for each fold are: {scores}')

        #folds_annotations.to_csv(os.path.join(saving_path, f'{c}_all_folds_test.csv'), index=False)

        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        sns.kdeplot(data=folds_annotations, x='predicted_probability_SimCLR', hue=hue, palette=condition_palette, ax=ax, fill=True, alpha=0.3)
        plt.title(f'{c} classifier - Average AUC: {np.mean(scores):.2f} - Best AUC: {best_auc:.2f} - Batch {b}')
        plt.xlabel(f'Pr({c})')
        plt.show()